<a href="https://colab.research.google.com/github/jennifer-algabre/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
!git clone https://github.com/jennifer-algabre/flyrank-ml-internship.git
%cd flyrank-ml-internship

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 123, done.
remote: Counting objects: 100% (123/123), done.
remote: Compressing objects: 100% (94/94), done.
remote: Total 123 (delta 38), reused 78 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (123/123), 1.85 MiB | 23.66 MiB/s, done.
Resolving deltas: 100% (38/38), done.
/content/flyrank-ml-internship


In [3]:
!find . -name "*.csv"

./outputs/refresh_queue_sample.csv
./data/raw/content_refresh_anonymized.csv


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

The unit of analysis is one content page.

Each row represents one pseudonymized content page with aggregated search performance and content-related metrics over the recent observation window provided in the starter dataset. The dataset summarizes historical behavior rather than daily observations, making it suitable for content prioritization.

In [4]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")

df.head()

Rows: 30,000
Columns: 44


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## 2. Fields: feature / label / context / excluded

### Features
These observable signals are available before making a refresh decision and may help prioritize pages for review.

- impressions_90d
- clicks_90d
- ctr
- avg_position
- engagement_rate
- scroll_rate
- word_count
- content_age_days
- days_since_last_update
- search_volume
- competition
- content_type
- main_intent

### Label / Proxy
- Priority score (derived proxy for ranking pages)
- For exploratory analysis, trend_direction can also be used to identify declining pages, but it is not used as an input feature.

### Context
These fields help identify or describe records but are not used for prediction.

- content_id
- client_id
- impression_tier
- position_tier
- competition_level

### Excluded
The following fields are excluded because they leak information about the outcome or are not appropriate as predictive features.

- trend_direction — represents the observed outcome.
- trend_pct — directly measures the change used to determine trend direction and would introduce data leakage.
- ai_traffic_pct — not central to the current lane and may not generalize.

In [5]:
feature_cols = [
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "word_count",
    "content_age_days",
    "days_since_last_update",
    "search_volume",
    "competition",
    "content_type",
    "main_intent"
]

label_cols = ["trend_direction"]

context_cols = [
    "content_id",
    "client_id",
    "impression_tier",
    "position_tier",
    "competition_level"
]

excluded_cols = [
    "trend_pct",
    "ai_traffic_pct"
]

print("Feature columns:", len(feature_cols))
print(feature_cols)

print("\nLabel/Proxy:")
print(label_cols)

print("\nContext columns:")
print(context_cols)

print("\nExcluded columns:")
print(excluded_cols)

Feature columns: 13
['impressions_90d', 'clicks_90d', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'word_count', 'content_age_days', 'days_since_last_update', 'search_volume', 'competition', 'content_type', 'main_intent']

Label/Proxy:
['trend_direction']

Context columns:
['content_id', 'client_id', 'impression_tier', 'position_tier', 'competition_level']

Excluded columns:
['trend_pct', 'ai_traffic_pct']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

The following queries verify the unit of analysis, dataset size, missing values, and the fields selected for this lane.

In [6]:
# Dataset shape
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")

Rows: 30,000
Columns: 44


In [7]:
# Verify one row represents one content page
print("Unique content IDs:", df["content_id"].nunique())
print("Total rows:", len(df))
print("One row per content page:",
      df["content_id"].nunique() == len(df))

Unique content IDs: 30000
Total rows: 30000
One row per content page: True


In [8]:
# Missing values in selected feature columns
feature_cols = [
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "word_count",
    "content_age_days",
    "days_since_last_update",
    "search_volume",
    "competition",
    "content_type",
    "main_intent"
]

df[feature_cols].isnull().sum().sort_values(ascending=False)

,0
word_count,7699
search_volume,2468
competition,2468
main_intent,2374
scroll_rate,125
engagement_rate,0
avg_position,0
ctr,0
clicks_90d,0
impressions_90d,0


In [9]:
# Verify the label distribution
df["trend_direction"].value_counts()

,count
trend_direction,
down,16262
stable,5962
up,4388
new,2236
flat,1152


In [10]:
# Verify the dataset window by inspecting 90-day metrics
window_cols = [
    "impressions_90d",
    "clicks_90d"
]

print(df[window_cols].describe())

       impressions_90d    clicks_90d
count     30000.000000  30000.000000
mean       5200.366300     16.097333
std       16838.019547     75.076958
min           1.000000      0.000000
25%          81.000000      0.000000
50%         731.000000      1.000000
75%        3615.250000      7.000000
max      517715.000000   4178.000000


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This dataset supports decision-making for content refresh prioritization, but it has several limitations.

The data is aggregated rather than daily, so it cannot explain exactly when a page's performance changed. The starter dataset also contains pseudonymized content and client identifiers, meaning it cannot be linked back to real websites or business context.

The label used for this project is based on observed trend direction, which supports ranking and prioritization rather than establishing cause-and-effect relationships. In addition, some feature columns contain missing values, so preprocessing may be required before model training.

The results should therefore be interpreted as decision support based on observed search performance signals, not as proof that a specific factor causes a page to improve or decline.